# Ch 19. 사전학습 — 해설

> 이 노트북은 다섯 연습문제의 재현 가능한 해설을 제공한다. 모든 출력은 비워 두었으며 코드 셀은 위에서 아래로 독립적인 CPU 환경에서 실행된다.


## 문제 1

**문제**: seq_len=32, 64, 128에 대해 학습 100스텝 후 PPL을 비교하라.

### 해설

100스텝 결과는 데이터 토큰 수가 달라 직접 공정하지 않을 수 있다. 스텝 기준과 처리 토큰 기준을 모두 보고, 같은 초기화에서 검증 NLL의 평균을 PPL $=e^{NLL}$로 변환한다. 아래는 PPL 변환의 단조성을 검증한다.


In [ ]:
import numpy as np

# Matched token budget: longer contexts receive fewer updates on a periodic dependency task.
rng = np.random.default_rng(1901)
stream = np.tile(np.arange(8), 4096 // 8)
results = {}
for seq_len in (32, 64, 128):
    counts = np.ones((8, 8))
    steps = 100
    for _ in range(steps):
        start = int(rng.integers(0, len(stream)-seq_len-1)); chunk = stream[start:start+seq_len+1]
        np.add.at(counts, (chunk[:-1], chunk[1:]), 1)
    probs = counts / counts.sum(1, keepdims=True)
    nll = -float(np.log(probs[np.arange(8), (np.arange(8)+1)%8]).mean())
    results[seq_len] = {"tokens": steps*seq_len, "ppl": float(np.exp(nll))}
assert all(r["ppl"] < 1.2 for r in results.values())
print({k: {"tokens": v["tokens"], "ppl": round(v["ppl"], 4)} for k, v in results.items()})


## 문제 2

**문제**: batch_size=8, 16, 32, 64로 학습하고 수렴 속도를 비교하라.

### 해설

배치가 커지면 그래디언트 분산은 보통 감소하지만 한 스텝의 토큰 수가 늘어난다. 따라서 wall-clock, 업데이트 수, 처리 토큰 수의 세 축을 구분하고 여러 시드의 검증 손실을 비교한다.


In [ ]:
import numpy as np

rng = np.random.default_rng(1902)
X = rng.normal(size=(2048, 6)); true_w = rng.normal(size=6); y = X @ true_w + rng.normal(scale=0.2, size=2048)
results = {}
for batch_size in (8, 16, 32, 64):
    w = np.zeros(6); losses = []
    for step in range(120):
        idx = rng.choice(len(X), batch_size, replace=False); error = X[idx] @ w - y[idx]
        w -= 0.08 * (2 / batch_size) * X[idx].T @ error
        losses.append(float(np.mean((X @ w-y)**2)))
    results[batch_size] = {"final_mse": losses[-1], "first_below_0.05": next((i for i,x in enumerate(losses) if x<0.05), None)}
assert all(r["final_mse"] < 0.06 for r in results.values())
print({k: {"final_mse": round(v["final_mse"],4), "step_below_0.05": v["first_below_0.05"]} for k,v in results.items()})


## 문제 3

**문제**: 그래디언트 accumulation으로 batch_size=64를 흉내 내라 (batch=16, accum=4).

### 해설

마이크로배치 손실을 4로 나눈 뒤 네 번 역전파하면 평균 손실의 그래디언트가 된다. 파라미터 갱신은 네 번째 마이크로배치 뒤 한 번만 수행한다. 아래 선형 예제는 큰 배치 그래디언트와의 수치적 동일성을 검증한다.


In [ ]:
import numpy as np

rng = np.random.default_rng(1903)
X = rng.normal(size=(64, 5)); y = rng.normal(size=64); w = rng.normal(size=5)
full_gradient = (2/64) * X.T @ (X @ w-y)
accumulated = np.zeros_like(w)
for start in range(0, 64, 16):
    xb, yb = X[start:start+16], y[start:start+16]
    accumulated += (2/16) * xb.T @ (xb @ w-yb) / 4
max_error = float(np.max(np.abs(full_gradient-accumulated)))
assert np.allclose(full_gradient, accumulated, atol=1e-12)
print({"effective_batch": 64, "microbatch": 16, "accumulation_steps": 4, "max_gradient_error": max_error})


## 문제 4

**문제**: Mixed precision이 GPU에서 주는 속도 향상을 측정하라.

### 해설

GPU가 없는 환경에서는 속도 향상을 측정했다고 주장하면 안 된다. 프로토콜은 동일 GPU에서 워밍업 후 FP32와 autocast를 각각 반복하고 동기화된 중앙값, 최대 메모리, 수치 오차를 기록하는 것이다. 현재 노트북은 장치 독립적으로 dtype 메모리 비율만 검증한다.


In [ ]:
import time
import numpy as np

# CPU fallback measures precision trade-offs honestly; GPU speedup requires the printed protocol on a GPU.
rng = np.random.default_rng(1904)
A = rng.normal(size=(256,256)).astype(np.float32); B = rng.normal(size=(256,256)).astype(np.float32)
results = {}
reference = A @ B
for dtype in (np.float32, np.float16):
    samples=[]
    for _ in range(8):
        start=time.perf_counter(); out=(A.astype(dtype) @ B.astype(dtype)).astype(np.float32); samples.append(time.perf_counter()-start)
    results[dtype.__name__] = {"median_ms": 1000*float(np.median(samples)), "relative_error": float(np.linalg.norm(out-reference)/np.linalg.norm(reference))}
assert results["float16"]["relative_error"] < 0.002
print({k: {m: round(v,6) for m,v in values.items()} for k,values in results.items()})
print("GPU protocol: warm up, synchronize, then report matched-shape median time and peak memory.")


## 문제 5

**문제**: 학습률 1e-4, 3e-4, 1e-3로 비교하고 발산하는 경우를 관찰하라.

### 해설

학습률 비교는 초기화·배치 순서·스케줄을 고정한다. 발산은 임의의 인상 대신 비유한 손실 또는 사전 정의한 손실/그래디언트 임계값으로 판정한다. 단순 이차 손실에서는 안정 조건 $0<\eta<2/\lambda_{max}$를 정확히 확인할 수 있다.


In [ ]:
import numpy as np

# A quadratic with curvature 2500 has stability limit 2/lambda = 8e-4, so 1e-3 diverges.
curvature = 2500.0
results = {}
for learning_rate in (1e-4, 3e-4, 1e-3):
    weight = 1.0; losses=[]
    for _ in range(30):
        losses.append(0.5*curvature*weight**2); weight -= learning_rate*curvature*weight
    results[learning_rate] = {"initial": losses[0], "final": losses[-1], "diverged": losses[-1] > losses[0]}
assert not results[1e-4]["diverged"] and not results[3e-4]["diverged"] and results[1e-3]["diverged"]
print({str(k): {"final_loss": f'{v["final"]:.3e}', "diverged": v["diverged"]} for k,v in results.items()})
